In [1]:
import sys
sys.path.insert(0, "../src")

import torch
from torch import nn
from torch.utils.data import DataLoader, random_split
from torchvision import datasets, transforms
from pathlib import Path

from femuralabs.toolkit.model import SmallCNN
from femuralabs.toolkit.train_utils import (
    set_seed, get_device, train_one_epoch, evaluate_with_accuracy,
    read_metrics, plot_curves,
)


In [2]:
set_seed(0)
device = get_device()
print("device:", device)

tfm = transforms.Compose([transforms.ToTensor(),
                          transforms.Normalize((0.1307,), (0.3081,))])
full = datasets.MNIST("../data", train=True, download=True, transform=tfm)
gen = torch.Generator().manual_seed(0)
train_set, val_set = random_split(full, [54000, 6000], generator=gen)

train_loader = DataLoader(train_set, batch_size=64, shuffle=True,
                          num_workers=0, generator=gen)
val_loader = DataLoader(val_set, batch_size=256, shuffle=False, num_workers=0)

model = SmallCNN().to(device)
loss_fn = nn.CrossEntropyLoss()
optimizer = torch.optim.SGD(model.parameters(), lr=0.01, momentum=0.9)

for epoch in range(1, 4):
    train_loss = train_one_epoch(model, train_loader, loss_fn, optimizer, device)
    val_loss, acc = evaluate_with_accuracy(model, val_loader, loss_fn, device)
    print(f"epoch {epoch}: train {train_loss:.4f} val {val_loss:.4f} acc {acc:.4f}")

device: cuda
epoch 1: train 0.2227 val 0.0884 acc 0.9722
epoch 2: train 0.0724 val 0.0620 acc 0.9813
epoch 3: train 0.0583 val 0.0592 acc 0.9795


In [3]:
csv_path = Path("../runs/run_a/metrics.csv")
if csv_path.exists():
    metrics = read_metrics(csv_path)
    print("epochs:", metrics["epoch"])
    print("train loss:", [round(v, 4) for v in metrics["train_loss"]])
    print("val loss:  ", [round(v, 4) for v in metrics["val_loss"]])
    plot_curves(csv_path, Path("../reports/notebook_check.png"))
else:
    print("run_a metrics not found; run the CLI script first")

epochs: [1.0, 2.0, 3.0, 4.0, 5.0]
train loss: [0.2227, 0.0724, 0.0583, 0.0502, 0.0446]
val loss:   [0.0884, 0.062, 0.0592, 0.0499, 0.055]
